In [ ]:

import os
import random
from pathlib import Path

import numpy as np
import torch

from src.other_models.diffusion_cond_ext import denormalize, fit_normalizer, normalize
from src.volsurface_latentFM.ivs_autoencoder import IVS_AE
from src.volsurface_latentFM.latent_models_variants import LatentFlowMatch


# REPRODUCIBILITY OF ABLATION STUDY

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

torch.use_deterministic_algorithms(True, warn_only=True)


def set_all_seeds(seed):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


# ============================================================
# PATHS AND CONTROLS 
# ============================================================

CUBE_PATH = Path("data/treated/vol_surface_spx.npz")
OUT_DIR = Path("data/treated/selected_retrain")
FIG_DIR = Path("figs/selected_retrain")

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not CUBE_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CUBE_PATH}; build it with constructor_surface_cube.py."
    )

AE_BASE_PARAMS = {
    "hidden_enc": (256, 128, 64),
    "hidden_dec": (64, 128, 256),
    "beta": 1e-2,
    "lambda_cal": 3e-2,
    "lambda_but": 2e-3,
}
AE_FINAL_FIT = {
    "batch_size": 64,
    "lr": 2e-3,
    "warmup_epochs": 200,
}
FM_FINAL_FIT = {
    "batch_size": 64,
    "lr": 1e-4,
}

LATENT_HPO_AE_EPOCHS = 1000
LATENT_HPO_FM_EPOCHS = 400
LEGACY_ARB_TOL = 1e-4

LATENT_SUMMARY_CSV = OUT_DIR / "latent_dimension_optuna_summary.csv"


# ============================================================
# DATA 
# ============================================================

d = np.load(CUBE_PATH, allow_pickle=True)

cube = np.asarray(d["cube"], dtype=np.float32)
m_new = np.asarray(d["m_grid"], dtype=float).ravel()
tenor_days = np.asarray(d["tenor_days"], dtype=float).ravel()
tau_new = tenor_days / 365.0

if cube.shape[:2] == (len(m_new), len(tenor_days)):
    surfaces = cube.transpose(2, 1, 0)
elif cube.shape[1:] == (len(tenor_days), len(m_new)):
    surfaces = cube
else:
    raise ValueError(f"Unexpected cube orientation: {cube.shape}")

if np.nanmedian(surfaces) < 2.0:
    surfaces = surfaces * 100.0

real_iv = torch.tensor(
    surfaces,
    dtype=torch.float32,
    device=device,
).unsqueeze(1)

NT, NM = real_iv.shape[-2:]

if (NT, NM) != (len(tau_new), len(m_new)):
    raise ValueError("Surface shape and stored grids do not agree.")

print("Cube:", CUBE_PATH, cube.shape)
print("Training tensor:", tuple(real_iv.shape))
print("Grid:", NT, "tenors x", NM, "moneyness points")
print("Device:", device)


# ============================================================
# HPO SPLIT 
# ============================================================

real_hpo_train = real_iv
real_hpo_val = real_iv

lo_hpo, hi_hpo = fit_normalizer(real_hpo_train)
surf_hpo_train_n = normalize(real_hpo_train, lo_hpo, hi_hpo)
surf_hpo_val_n = normalize(real_hpo_val, lo_hpo, hi_hpo)
lo_hpo_v = float(lo_hpo.item())
hi_hpo_v = float(hi_hpo.item())


# ============================================================
# SELECTED LATENT DIMENSION
# ============================================================
# Pull BEST_Z_DIM from the Optuna summary 

DEFAULT_BEST_Z_DIM = 10

if LATENT_SUMMARY_CSV.exists():
    import pandas as pd

    _latent_summary_df = pd.read_csv(LATENT_SUMMARY_CSV)
    BEST_Z_DIM = int(
        _latent_summary_df.loc[_latent_summary_df["score_mean"].idxmin(), "z_dim"]
    )
    print(f"Loaded BEST_Z_DIM={BEST_Z_DIM} from {LATENT_SUMMARY_CSV}")
else:
    BEST_Z_DIM = DEFAULT_BEST_Z_DIM
    print(
        f"{LATENT_SUMMARY_CSV} not found -- using DEFAULT_BEST_Z_DIM={BEST_Z_DIM}. "
        "Run the Optuna cell in Paper_illustration first if you want the ablation "
        "to match its actually-selected z_dim."
    )


# ============================================================
# METRIC HELPERS 
# ============================================================

def sliced_wasserstein(X, Y, n_proj=150, seed=42):
    """Sliced Wasserstein distance for two sample matrices."""
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)

    if X.ndim != 2 or Y.ndim != 2 or X.shape[1] != Y.shape[1]:
        raise ValueError(
            f"Expected two matrices with the same feature dimension, got {X.shape} and {Y.shape}."
        )

    rng = np.random.default_rng(int(seed))
    directions = rng.standard_normal((int(n_proj), X.shape[1]))
    directions /= np.maximum(
        np.linalg.norm(directions, axis=1, keepdims=True),
        1e-12,
    )

    proj_x = np.sort(X @ directions.T, axis=0)
    proj_y = np.sort(Y @ directions.T, axis=0)

    n_common = max(len(proj_x), len(proj_y))
    u_common = np.linspace(0.0, 1.0, n_common)
    u_x = np.linspace(0.0, 1.0, len(proj_x))
    u_y = np.linspace(0.0, 1.0, len(proj_y))

    distances = []
    for j in range(int(n_proj)):
        x_j = np.interp(u_common, u_x, proj_x[:, j])
        y_j = np.interp(u_common, u_y, proj_y[:, j])
        distances.append(np.mean(np.abs(x_j - y_j)))

    return float(np.mean(distances))


def decode_with_ae(ae_model, latent_codes, lo_value, hi_value):
    """Decode latent codes and return IV surfaces in percentage units."""
    decoded_n = (
        ae_model.decode(latent_codes, device=device)
        .reshape(-1, NT, NM)
        .cpu()
    )
    lo_tensor = torch.tensor(float(lo_value), dtype=torch.float32)
    hi_tensor = torch.tensor(float(hi_value), dtype=torch.float32)
    return denormalize(decoded_n, lo_tensor, hi_tensor).numpy()


def legacy_phi_batch(surfaces_fraction):
    """
    Historical arbitrage score used in the earlier model comparisons.

    This diagnostic deliberately uses the old linear comparison axes,
    old relative-call transformation and unscaled butterfly second difference.
    It is used only for the Optuna score/reporting in this paper notebook.
    """
    from scipy.stats import norm

    sigma = np.asarray(surfaces_fraction, dtype=float)
    if sigma.ndim != 3 or sigma.shape[1:] != (NT, NM):
        raise ValueError(
            f"Expected surfaces with shape (n, {NT}, {NM}), got {sigma.shape}."
        )

    m_legacy = np.linspace(float(np.min(m_new)), float(np.max(m_new)), NM)
    tau_legacy = np.linspace(float(np.min(tau_new)), float(np.max(tau_new)), NT)

    T = tau_legacy[None, :, None]
    M = m_legacy[None, None, :]
    vol = np.clip(sigma, 1e-8, None)
    T = np.clip(T, 1e-12, None)

    d1 = (-M + 0.5 * vol**2 * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    calls = norm.cdf(d1) - np.exp(M) * norm.cdf(d2)
    calls = np.clip(calls, 1e-12, None)

    dm = np.diff(m_legacy)
    dt = np.diff(tau_legacy)

    dC_dm = np.diff(calls, axis=2) / dm[None, None, :]
    p_call = np.maximum(dC_dm, 0.0).sum(axis=(1, 2))

    d2C = calls[:, :, 2:] - 2.0 * calls[:, :, 1:-1] + calls[:, :, :-2]
    p_bfly = np.maximum(-d2C, 0.0).sum(axis=(1, 2))

    dC_dt = np.diff(calls, axis=1) / dt[None, :, None]
    p_calendar = np.maximum(-dC_dt, 0.0).sum(axis=(1, 2))

    return p_call + p_bfly + p_calendar


print("Ablation setup ready: BEST_Z_DIM =", BEST_Z_DIM)


In [ ]:

import copy
import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from IPython.display import display
from scipy.stats import norm, wasserstein_distance
from torch.utils.data import DataLoader, TensorDataset


# ============================================================
# CONTROLS
# ============================================================

RUN_LFM_ABLATION = True

if not RUN_LFM_ABLATION:
    raise RuntimeError(
        "Checkpoint-independent mode requires RUN_LFM_ABLATION=True."
    )

ABLATION_SEEDS = (42, 43, 44)

ABLATION_AE_EPOCHS = int(LATENT_HPO_AE_EPOCHS)
ABLATION_FM_EPOCHS = int(LATENT_HPO_FM_EPOCHS)

ABLATION_N_GEN = 512
ABLATION_N_PROJ = 300
ABLATION_SWD_SEED = 123456

# The original L-FM has lambda_call = 0.
# This weight is used only for the additional call-spread experiment.
FULL_LAMBDA_CALL = 3e-2

CALL_SPREAD_TOL = 1e-8

ABLATION_DIR = Path(OUT_DIR) / "lfm_ablation"
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

ABLATION_TRIALS_PATH = ABLATION_DIR / "lfm_penalty_ablation_trials.csv"
ABLATION_SUMMARY_PATH = ABLATION_DIR / "lfm_penalty_ablation_summary.csv"


# ============================================================
# CHECK PREVIOUS NOTEBOOK CELLS
# ============================================================

REQUIRED_OBJECTS = [
    "BEST_Z_DIM",
    "AE_BASE_PARAMS",
    "AE_FINAL_FIT",
    "FM_FINAL_FIT",
    "surf_hpo_train_n",
    "surf_hpo_val_n",
    "real_hpo_train",
    "real_hpo_val",
    "lo_hpo_v",
    "hi_hpo_v",
    "m_new",
    "tau_new",
    "NT",
    "NM",
    "device",
    "set_all_seeds",
    "sliced_wasserstein",
    "decode_with_ae",
    "legacy_phi_batch",
    "LEGACY_ARB_TOL",
    "IVS_AE",
    "LatentFlowMatch",
    "OUT_DIR",
    "FIG_DIR",
]

missing_objects = [name for name in REQUIRED_OBJECTS if name not in globals()]

if missing_objects:
    raise NameError(
        "Run the preceding Paper_illustration cells first. "
        f"Missing objects: {', '.join(missing_objects)}"
    )


# ============================================================
# TENSOR HELPERS
# ============================================================

def to_flat_tensor(surfaces):
    """
    Convert surfaces with shape:
        (N, 1, NT, NM),
        (N, NT, NM),
    or:
        (N, NT * NM)

    into a flat float tensor on the selected device.
    """
    if isinstance(surfaces, torch.Tensor):
        tensor = surfaces.detach().to(device=device, dtype=torch.float32)
    else:
        tensor = torch.as_tensor(
            surfaces,
            dtype=torch.float32,
            device=device,
        )

    if tensor.ndim == 4:
        tensor = tensor[:, 0]

    if tensor.ndim not in (2, 3):
        raise ValueError(f"Unexpected surface shape: {tuple(tensor.shape)}")

    return tensor.reshape(len(tensor), -1).contiguous()


def decoded_surface_pct(x_hat, lo, hi):
    """
    Convert the decoded normalised representation back to
    implied-volatility percentage units.
    """
    surface_n = x_hat.reshape(-1, NT, NM)

    lo_tensor = torch.as_tensor(
        lo,
        dtype=surface_n.dtype,
        device=surface_n.device,
    )

    hi_tensor = torch.as_tensor(
        hi,
        dtype=surface_n.dtype,
        device=surface_n.device,
    )

    return (surface_n + 1.0) / 2.0 * (hi_tensor - lo_tensor) + lo_tensor


# ============================================================
# FUNCTIONAL LOSS TERMS
# ============================================================

def calendar_loss(x_hat, lo, hi):
    """
    Calendar proxy used by the existing autoencoder:
        sigma(T_{j+1}, m) >= sigma(T_j, m).
    """
    surface_pct = decoded_surface_pct(x_hat, lo, hi)

    maturity_difference = (
        surface_pct[:, 1:, :]
        - surface_pct[:, :-1, :]
    )

    return F.relu(-maturity_difference).mean()


def butterfly_loss(x_hat, lo, hi):
    """
    Gatheral g(k) butterfly penalty used by the existing AE.
    """
    surface_pct = decoded_surface_pct(x_hat, lo, hi)
    sigma = surface_pct / 100.0

    m_grid_tensor = torch.as_tensor(
        m_new,
        dtype=surface_pct.dtype,
        device=surface_pct.device,
    )

    tau_grid_tensor = torch.as_tensor(
        tau_new,
        dtype=surface_pct.dtype,
        device=surface_pct.device,
    )

    total_variance = sigma.pow(2) * tau_grid_tensor[None, :, None]

    log_moneyness = torch.log(
        torch.clamp(m_grid_tensor, min=1e-8)
    )

    dk = (
        log_moneyness[-1]
        - log_moneyness[0]
    ) / max(len(log_moneyness) - 1, 1)

    dk = torch.clamp(dk, min=1e-8)

    w_left = total_variance[:, :, :-2]
    w_middle = total_variance[:, :, 1:-1]
    w_right = total_variance[:, :, 2:]

    w_first_derivative = (w_right - w_left) / (2.0 * dk)
    w_second_derivative = (
        w_right
        - 2.0 * w_middle
        + w_left
    ) / (dk**2)

    k_middle = log_moneyness[1:-1][None, None, :]
    w_safe = torch.clamp(w_middle, min=1e-4)

    gatheral_g = (
        (
            1.0
            - k_middle * w_first_derivative / (2.0 * w_safe)
        ).pow(2)
        - (
            w_first_derivative.pow(2) / 4.0
        ) * (
            1.0 / w_safe + 0.25
        )
        + 0.5 * w_second_derivative
    )

    return F.relu(-gatheral_g).mean()


def call_spread_loss(x_hat, lo, hi):
    """
    Differentiable call-price monotonicity penalty.

    With ordinary moneyness m = K/S and zero carry:

        c(m,T) = N(d1) - m N(d2).

    A valid call surface must satisfy:

        dc/dm <= 0.
    """
    surface_pct = decoded_surface_pct(x_hat, lo, hi)

    sigma = torch.clamp(
        surface_pct / 100.0,
        min=1e-6,
        max=5.0,
    )

    m_grid_tensor = torch.as_tensor(
        m_new,
        dtype=surface_pct.dtype,
        device=surface_pct.device,
    )[None, None, :]

    tau_grid_tensor = torch.as_tensor(
        tau_new,
        dtype=surface_pct.dtype,
        device=surface_pct.device,
    )[None, :, None]

    m_grid_tensor = torch.clamp(m_grid_tensor, min=1e-8)
    tau_grid_tensor = torch.clamp(tau_grid_tensor, min=1e-10)

    total_variance = sigma.pow(2) * tau_grid_tensor

    sqrt_total_variance = torch.sqrt(
        torch.clamp(total_variance, min=1e-12)
    )

    d1 = (
        -torch.log(m_grid_tensor)
        + 0.5 * total_variance
    ) / sqrt_total_variance

    d2 = d1 - sqrt_total_variance

    normal_cdf_d1 = 0.5 * (
        1.0
        + torch.erf(d1 / np.sqrt(2.0))
    )

    normal_cdf_d2 = 0.5 * (
        1.0
        + torch.erf(d2 / np.sqrt(2.0))
    )

    calls = (
        normal_cdf_d1
        - m_grid_tensor * normal_cdf_d2
    )

    dm = (
        m_grid_tensor[:, :, 1:]
        - m_grid_tensor[:, :, :-1]
    )

    call_slopes = (
        calls[:, :, 1:]
        - calls[:, :, :-1]
    ) / dm

    return F.relu(call_slopes).mean()


def functional_ae_loss(
    x,
    x_hat,
    mu,
    logvar,
    beta_now,
    weights,
    lo,
    hi,
):
    """
    Complete functional AE objective.
    """
    reconstruction = F.mse_loss(x_hat, x)

    kl = -0.5 * torch.mean(
        1.0
        + logvar
        - mu.pow(2)
        - logvar.exp()
    )

    calendar = calendar_loss(x_hat, lo, hi)
    butterfly = butterfly_loss(x_hat, lo, hi)
    call_spread = call_spread_loss(x_hat, lo, hi)

    total = (
        reconstruction
        + float(beta_now) * kl
        + float(weights["lambda_cal"]) * calendar
        + float(weights["lambda_but"]) * butterfly
        + float(weights["lambda_call"]) * call_spread
    )

    return (
        total,
        reconstruction,
        kl,
        calendar,
        butterfly,
        call_spread,
    )


# ============================================================
# FUNCTIONAL AE TRAINING LOOP
# ============================================================

def train_ae_functionally(weights, seed):
    """
    Instantiate the existing IVS_AE architecture and train it
    with the functional objective above.

    IVS_AE.fit() and IVS_AE.loss() are not used.
    """
    set_all_seeds(seed)

    model = IVS_AE(
        z_dim=int(BEST_Z_DIM),
        hidden_enc=AE_BASE_PARAMS["hidden_enc"],
        hidden_dec=AE_BASE_PARAMS["hidden_dec"],
        beta=float(weights["beta"]),
        lambda_cal=float(weights["lambda_cal"]),
        lambda_but=float(weights["lambda_but"]),
    ).to(device)

    x_train = to_flat_tensor(surf_hpo_train_n)
    x_validation = to_flat_tensor(surf_hpo_val_n)

    loader = DataLoader(
        TensorDataset(x_train),
        batch_size=int(AE_FINAL_FIT["batch_size"]),
        shuffle=True,
        drop_last=False,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(AE_FINAL_FIT["lr"]),
    )

    warmup_epochs = min(
        int(AE_FINAL_FIT["warmup_epochs"]),
        max(1, ABLATION_AE_EPOCHS // 3),
    )

    history = {
        key: []
        for key in (
            "loss",
            "reconstruction",
            "kl",
            "calendar",
            "butterfly",
            "call_spread",
            "validation_loss",
        )
    }

    best_validation = float("inf")
    best_state = None

    for epoch in range(1, ABLATION_AE_EPOCHS + 1):
        beta_now = float(weights["beta"]) * min(
            1.0,
            epoch / max(warmup_epochs, 1),
        )

        model.train()

        epoch_sums = {
            key: 0.0
            for key in (
                "loss",
                "reconstruction",
                "kl",
                "calendar",
                "butterfly",
                "call_spread",
            )
        }

        for (batch,) in loader:
            optimizer.zero_grad(set_to_none=True)

            x_hat, mu, logvar = model(batch)

            (
                total,
                reconstruction,
                kl,
                calendar,
                butterfly,
                call_spread,
            ) = functional_ae_loss(
                x=batch,
                x_hat=x_hat,
                mu=mu,
                logvar=logvar,
                beta_now=beta_now,
                weights=weights,
                lo=lo_hpo_v,
                hi=hi_hpo_v,
            )

            total.backward()
            optimizer.step()

            values = {
                "loss": total,
                "reconstruction": reconstruction,
                "kl": kl,
                "calendar": calendar,
                "butterfly": butterfly,
                "call_spread": call_spread,
            }

            for key, value in values.items():
                epoch_sums[key] += float(
                    value.detach().cpu()
                )

        number_batches = max(len(loader), 1)

        for key in epoch_sums:
            history[key].append(
                epoch_sums[key] / number_batches
            )

        model.eval()

        with torch.no_grad():
            (
                x_hat_validation,
                mu_validation,
                logvar_validation,
            ) = model(x_validation)

            validation_total, _, _, _, _, _ = functional_ae_loss(
                x=x_validation,
                x_hat=x_hat_validation,
                mu=mu_validation,
                logvar=logvar_validation,
                beta_now=beta_now,
                weights=weights,
                lo=lo_hpo_v,
                hi=hi_hpo_v,
            )

            validation_loss = float(
                validation_total.detach().cpu()
            )

        history["validation_loss"].append(validation_loss)

        if validation_loss < best_validation - 1e-7:
            best_validation = validation_loss

            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print_interval = max(
            1,
            ABLATION_AE_EPOCHS // 3,
        )
        if (
            epoch == 1
            or epoch == ABLATION_AE_EPOCHS
            or epoch % print_interval == 0
        ):
            print(
                f"  AE {epoch:>4}/{ABLATION_AE_EPOCHS} | "
                f"recon={history['reconstruction'][-1]:.5f} | "
                f"KL={history['kl'][-1]:.5f} | "
                f"cal={history['calendar'][-1]:.5f} | "
                f"but={history['butterfly'][-1]:.5f} | "
                f"call={history['call_spread'][-1]:.5f} | "
                f"val={validation_loss:.5f}"
            )

    if best_state is not None:
        model.load_state_dict(
            {
                key: value.to(device)
                for key, value in best_state.items()
            }
        )

    model.eval()

    return model, history


def encode_mean(model, surfaces):
    """
    Return the deterministic encoder mean.
    """
    x = to_flat_tensor(surfaces)

    model.eval()

    with torch.no_grad():
        mu, _ = model.encoder(x)

    return mu.detach().cpu().numpy()


# ============================================================
# CALL-SPREAD DIAGNOSTIC
# ============================================================

def call_spread_diagnostics(surfaces_fraction):
    """
    Call-price monotonicity diagnostic on generated surfaces.
    """
    sigma = np.asarray(
        surfaces_fraction,
        dtype=np.float64,
    )

    if sigma.ndim != 3 or sigma.shape[1:] != (NT, NM):
        raise ValueError(
            f"Expected shape (N, {NT}, {NM}), obtained {sigma.shape}."
        )

    m_grid_array = np.asarray(
        m_new,
        dtype=np.float64,
    )[None, None, :]

    tau_grid_array = np.asarray(
        tau_new,
        dtype=np.float64,
    )[None, :, None]

    sigma = np.clip(sigma, 1e-8, 5.0)

    total_variance = sigma**2 * tau_grid_array

    sqrt_total_variance = np.sqrt(
        np.maximum(total_variance, 1e-16)
    )

    d1 = (
        -np.log(m_grid_array)
        + 0.5 * total_variance
    ) / sqrt_total_variance

    d2 = d1 - sqrt_total_variance

    calls = (
        norm.cdf(d1)
        - m_grid_array * norm.cdf(d2)
    )

    call_slopes = (
        np.diff(calls, axis=2)
        / np.diff(m_grid_array, axis=2)
    )

    violations = np.maximum(
        call_slopes,
        0.0,
    )

    mean_violation = violations.mean(axis=(1, 2))
    max_violation = violations.max(axis=(1, 2))

    return {
        "mean": mean_violation,
        "max": max_violation,
        "clean": max_violation <= CALL_SPREAD_TOL,
    }


# ============================================================
# ABLATION CONFIGURATIONS
# ============================================================

FULL_BETA = float(AE_BASE_PARAMS["beta"])
FULL_LAMBDA_CAL = float(AE_BASE_PARAMS["lambda_cal"])
FULL_LAMBDA_BUT = float(AE_BASE_PARAMS["lambda_but"])

ABLATIONS = {
    "Original full": {
        "beta": FULL_BETA,
        "lambda_cal": FULL_LAMBDA_CAL,
        "lambda_but": FULL_LAMBDA_BUT,
        "lambda_call": 0.0,
    },

    "Without KL": {
        "beta": 0.0,
        "lambda_cal": FULL_LAMBDA_CAL,
        "lambda_but": FULL_LAMBDA_BUT,
        "lambda_call": 0.0,
    },

    "Without calendar": {
        "beta": FULL_BETA,
        "lambda_cal": 0.0,
        "lambda_but": FULL_LAMBDA_BUT,
        "lambda_call": 0.0,
    },

    "Without butterfly": {
        "beta": FULL_BETA,
        "lambda_cal": FULL_LAMBDA_CAL,
        "lambda_but": 0.0,
        "lambda_call": 0.0,
    },

    "Original + call spread": {
        "beta": FULL_BETA,
        "lambda_cal": FULL_LAMBDA_CAL,
        "lambda_but": FULL_LAMBDA_BUT,
        "lambda_call": FULL_LAMBDA_CALL,
    },
}

ABLATION_ORDER = list(ABLATIONS)
REFERENCE_VARIANT = "Original full"

print("Ablation configurations")
display(pd.DataFrame(ABLATIONS).T)


# ============================================================
# HELD-OUT EVALUATION SAMPLE
# ============================================================

real_ablation_eval = (
    real_hpo_val[:, 0]
    .detach()
    .cpu()
    .numpy()
    / 100.0
).astype(np.float32)

print("Ablation training sample:", len(real_hpo_train))
print("Ablation validation sample:", len(real_hpo_val))
print("Selected latent dimension:", BEST_Z_DIM)


# ============================================================
# ONE COMPLETE AE + FM RUN
# ============================================================

def run_ablation(variant, weights, seed):
    print("\n" + "=" * 72)
    print(f"{variant} | seed={seed}")
    print("=" * 72)

    ae_model, ae_history = train_ae_functionally(
        weights=weights,
        seed=int(seed),
    )

    z_train = encode_mean(
        ae_model,
        surf_hpo_train_n,
    )

    # Paired FM initialisation across variants.
    set_all_seeds(int(seed) + 10_000)

    fm_model = LatentFlowMatch(
        z_dim=int(BEST_Z_DIM),
        path="trigo",
    )

    fm_model.fit(
        z_train,
        n_epochs=ABLATION_FM_EPOCHS,
        batch_size=int(FM_FINAL_FIT["batch_size"]),
        lr=float(FM_FINAL_FIT["lr"]),
        print_every=max(1, ABLATION_FM_EPOCHS),
        device=device,
    )

    # Paired sampling noise across variants.
    set_all_seeds(int(seed) + 20_000)

    latent_sample = fm_model.sample(
        ABLATION_N_GEN,
        device=device,
    )

    generated_fraction = (
        np.asarray(
            decode_with_ae(
                ae_model,
                latent_sample,
                lo_hpo_v,
                hi_hpo_v,
            ),
            dtype=np.float32,
        )
        / 100.0
    )

    global_w1 = float(
        wasserstein_distance(
            real_ablation_eval.reshape(-1),
            generated_fraction.reshape(-1),
        )
    )

    sliced_w1 = float(
        sliced_wasserstein(
            real_ablation_eval.reshape(
                len(real_ablation_eval),
                -1,
            ),
            generated_fraction.reshape(
                len(generated_fraction),
                -1,
            ),
            n_proj=ABLATION_N_PROJ,
            seed=ABLATION_SWD_SEED,
        )
    )

    legacy_phi = legacy_phi_batch(
        generated_fraction
    )

    call_diagnostics = call_spread_diagnostics(
        generated_fraction
    )

    result = {
        "variant": variant,
        "seed": int(seed),
        "z_dim": int(BEST_Z_DIM),

        "beta": float(weights["beta"]),
        "lambda_cal": float(weights["lambda_cal"]),
        "lambda_but": float(weights["lambda_but"]),
        "lambda_call": float(weights["lambda_call"]),

        "global_w1": global_w1,
        "sliced_w1": sliced_w1,

        "legacy_mean_phi": float(
            np.mean(legacy_phi)
        ),

        "legacy_zero_arb_pct": float(
            100.0
            * np.mean(
                np.abs(legacy_phi)
                < LEGACY_ARB_TOL
            )
        ),

        "call_spread_mean_violation": float(
            np.mean(
                call_diagnostics["mean"]
            )
        ),

        "call_spread_clean_pct": float(
            100.0
            * np.mean(
                call_diagnostics["clean"]
            )
        ),

        "ae_best_val_loss": float(
            np.min(
                ae_history["validation_loss"]
            )
        ),
    }

    print(
        f"W1={global_w1:.6f} | "
        f"SWD={sliced_w1:.6f} | "
        f"legacy arb-free={result['legacy_zero_arb_pct']:.1f}% | "
        f"call-clean={result['call_spread_clean_pct']:.1f}%"
    )

    del ae_model
    del fm_model
    del z_train
    del latent_sample
    del generated_fraction
    del legacy_phi
    del call_diagnostics

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# RUN OR LOAD
# ============================================================

if RUN_LFM_ABLATION:
    ablation_rows = [
        run_ablation(
            variant=variant,
            weights=ABLATIONS[variant],
            seed=seed,
        )
        for seed in ABLATION_SEEDS
        for variant in ABLATION_ORDER
    ]

    ablation_trials_df = pd.DataFrame(
        ablation_rows
    )



# ============================================================
# PAIRED DIFFERENCES AGAINST ORIGINAL FULL
# ============================================================

reference_results = (
    ablation_trials_df.loc[
        ablation_trials_df["variant"] == REFERENCE_VARIANT,
        ["seed", "global_w1", "sliced_w1"],
    ]
    .rename(
        columns={
            "global_w1": "reference_global_w1",
            "sliced_w1": "reference_sliced_w1",
        }
    )
)

ablation_trials_df = ablation_trials_df.merge(
    reference_results,
    on="seed",
    how="left",
    validate="many_to_one",
)

for metric in ("global_w1", "sliced_w1"):
    reference_column = f"reference_{metric}"
    delta_column = f"{metric}_delta"
    delta_pct_column = f"{metric}_delta_pct"

    ablation_trials_df[delta_column] = (
        ablation_trials_df[metric]
        - ablation_trials_df[reference_column]
    )

    ablation_trials_df[delta_pct_column] = (
        100.0
        * ablation_trials_df[delta_column]
        / np.maximum(
            np.abs(
                ablation_trials_df[
                    reference_column
                ]
            ),
            1e-12,
        )
    )

ablation_trials_df.to_csv(
    ABLATION_TRIALS_PATH,
    index=False,
)


# ============================================================
# SUMMARY ACROSS SEEDS
# ============================================================

ablation_summary_df = (
    ablation_trials_df
    .groupby(
        "variant",
        as_index=False,
    )
    .agg(
        global_w1_mean=("global_w1", "mean"),
        global_w1_std=("global_w1", "std"),

        sliced_w1_mean=("sliced_w1", "mean"),
        sliced_w1_std=("sliced_w1", "std"),

        global_w1_delta_pct_mean=(
            "global_w1_delta_pct",
            "mean",
        ),

        global_w1_delta_pct_std=(
            "global_w1_delta_pct",
            "std",
        ),

        sliced_w1_delta_pct_mean=(
            "sliced_w1_delta_pct",
            "mean",
        ),

        sliced_w1_delta_pct_std=(
            "sliced_w1_delta_pct",
            "std",
        ),

        legacy_mean_phi_mean=(
            "legacy_mean_phi",
            "mean",
        ),

        legacy_zero_arb_pct_mean=(
            "legacy_zero_arb_pct",
            "mean",
        ),

        call_spread_mean_violation_mean=(
            "call_spread_mean_violation",
            "mean",
        ),

        call_spread_clean_pct_mean=(
            "call_spread_clean_pct",
            "mean",
        ),

        ae_best_val_loss_mean=(
            "ae_best_val_loss",
            "mean",
        ),
    )
)

ablation_summary_df["variant"] = pd.Categorical(
    ablation_summary_df["variant"],
    categories=ABLATION_ORDER,
    ordered=True,
)

ablation_summary_df = (
    ablation_summary_df
    .sort_values("variant")
    .reset_index(drop=True)
)

numeric_columns = (
    ablation_summary_df
    .select_dtypes(include=[np.number])
    .columns
)

ablation_summary_df[numeric_columns] = (
    ablation_summary_df[numeric_columns]
    .fillna(0.0)
)

ablation_summary_df.to_csv(
    ABLATION_SUMMARY_PATH,
    index=False,
)


# ============================================================
# TABLES
# ============================================================

print("\nIndividual ablation runs")

display(
    ablation_trials_df.style.format(
        {
            "global_w1": "{:.6f}",
            "sliced_w1": "{:.6f}",
            "global_w1_delta_pct": "{:+.1f}%",
            "sliced_w1_delta_pct": "{:+.1f}%",
            "legacy_mean_phi": "{:.6f}",
            "legacy_zero_arb_pct": "{:.1f}%",
            "call_spread_mean_violation": "{:.3e}",
            "call_spread_clean_pct": "{:.1f}%",
            "ae_best_val_loss": "{:.6f}",
        }
    )
)

print("\nMean results across seeds")

display(
    ablation_summary_df.style.format(
        {
            "global_w1_mean": "{:.6f}",
            "global_w1_std": "{:.6f}",
            "sliced_w1_mean": "{:.6f}",
            "sliced_w1_std": "{:.6f}",
            "global_w1_delta_pct_mean": "{:+.1f}%",
            "global_w1_delta_pct_std": "{:.1f}%",
            "sliced_w1_delta_pct_mean": "{:+.1f}%",
            "sliced_w1_delta_pct_std": "{:.1f}%",
            "legacy_mean_phi_mean": "{:.6f}",
            "legacy_zero_arb_pct_mean": "{:.1f}%",
            "call_spread_mean_violation_mean": "{:.3e}",
            "call_spread_clean_pct_mean": "{:.1f}%",
            "ae_best_val_loss_mean": "{:.6f}",
        }
    )
)


# ============================================================
# FIGURE HELPER
# ============================================================

def plot_ablation_bars(
    mean_column,
    std_column,
    ylabel,
    title,
    filename,
):
    x_positions = np.arange(
        len(ablation_summary_df)
    )

    labels = (
        ablation_summary_df["variant"]
        .astype(str)
    )

    fig, ax = plt.subplots(
        figsize=(9, 5)
    )

    ax.bar(
        x_positions,
        ablation_summary_df[mean_column],
        yerr=ablation_summary_df[std_column],
        capsize=5,
        label="Mean ± 1 std",
    )

    ax.set_xticks(x_positions)

    ax.set_xticklabels(
        labels,
        rotation=18,
        ha="right",
    )

    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best")

    ax.grid(
        axis="y",
        alpha=0.3,
    )

    plt.tight_layout()

    plt.savefig(
        Path(FIG_DIR) / filename,
        dpi=180,
        bbox_inches="tight",
    )

    plt.show()


# ============================================================
# GLOBAL W1
# ============================================================

plot_ablation_bars(
    mean_column="global_w1_mean",
    std_column="global_w1_std",
    ylabel="Global W1",
    title="L-FM ablation: Global W1",
    filename="LFM_ablation_global_W1.png",
)


# ============================================================
# SLICED W1
# ============================================================

plot_ablation_bars(
    mean_column="sliced_w1_mean",
    std_column="sliced_w1_std",
    ylabel="Sliced W1",
    title="L-FM ablation: Sliced W1",
    filename="LFM_ablation_sliced_W1.png",
)


# ============================================================
# RELATIVE CHANGE AGAINST ORIGINAL FULL
# ============================================================

x_positions = np.arange(
    len(ablation_summary_df)
)

labels = (
    ablation_summary_df["variant"]
    .astype(str)
)

bar_width = 0.38

fig, ax = plt.subplots(
    figsize=(9.5, 5.2)
)

ax.bar(
    x_positions - bar_width / 2.0,
    ablation_summary_df[
        "global_w1_delta_pct_mean"
    ],
    width=bar_width,
    yerr=ablation_summary_df[
        "global_w1_delta_pct_std"
    ],
    capsize=4,
    label="Global W1",
)

ax.bar(
    x_positions + bar_width / 2.0,
    ablation_summary_df[
        "sliced_w1_delta_pct_mean"
    ],
    width=bar_width,
    yerr=ablation_summary_df[
        "sliced_w1_delta_pct_std"
    ],
    capsize=4,
    label="Sliced W1",
)

ax.axhline(
    0.0,
    linewidth=1.0,
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    labels,
    rotation=18,
    ha="right",
)

ax.set_ylabel(
    "Change versus original full model (%)"
)

ax.set_title(
    "Contribution of each L-FM objective term"
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.3,
)

plt.tight_layout()

plt.savefig(
    Path(FIG_DIR)
    / "LFM_ablation_relative_degradation.png",
    dpi=180,
    bbox_inches="tight",
)

plt.show()


print("\nSaved:")
print(" ", ABLATION_TRIALS_PATH)
print(" ", ABLATION_SUMMARY_PATH)